# Inferencia CPU — Experimento B (dos etapas, ADR-006)

Demo reproducible del detector ganador **Experimento B / corrida T1** (real 459 + sintético 197, `imgsz=512`, seleccionado por mayor `val mAP50-95`) más la segunda etapa de **identificación de producto** por retrieval visual contra el catálogo (`src/recognition.py`, SIFT + BFMatcher, sin clasificación supervisada por SKU: ~1 muestra por SKU lo hace inviable — ver `docs/governance/03_PHASE2/14_ADR-006_...md`).

El notebook fuerza CPU, verifica el peso congelado por SHA-256, corre sobre escenas multiproducto sintéticas de ejemplo, identifica cada producto contra el catálogo y muestra boxes + nombre/SKU + confianza + similitud. No reentrena ni asume CUDA.


In [ ]:
import csv
import hashlib
import os
import sys
from pathlib import Path

os.environ['CUDA_VISIBLE_DEVICES'] = ''


def find_repo_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'src' / 'predict.py').is_file():
            return candidate
    raise RuntimeError('No se encontró la raíz del repositorio')


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))

# Rutas relativas por defecto (dentro del repo); override por variable de
# entorno para apuntar a los artefactos reales (pesos/catálogo/escenas viven
# fuera de Git por tamaño -- ver docs/governance -- nunca se hardcodea una
# ruta absoluta en este archivo).
MODEL_PATH = Path(os.environ.get(
    'AMARKET_MODEL_PATH',
    ROOT / 'outputs' / 'demo_model_b' / 'experiment_b_t1_best.pt',
))
CATALOG_PATH = Path(os.environ.get(
    'AMARKET_CATALOG_PATH',
    ROOT / 'outputs' / 'demo_model_b' / 'catalog_identity.csv',
))
DEMO_INPUT = Path(os.environ.get(
    'AMARKET_DEMO_INPUT',
    ROOT / 'outputs' / 'demo_model_b' / 'demo_scenes',
))
OUTPUT_DIR = ROOT / 'outputs' / 'notebook_cpu_demo_b'

EXPECTED_SHA256 = (
    '6635a3efa75da83cd53544281243359'
    'a68038966eede4f962ba60d1fa2de910b'
)

assert MODEL_PATH.is_file(), MODEL_PATH
assert CATALOG_PATH.is_file(), CATALOG_PATH
assert DEMO_INPUT.is_dir(), DEMO_INPUT

print('ROOT=', ROOT)
print('MODEL_PATH=', MODEL_PATH)
print('CATALOG_PATH=', CATALOG_PATH)
print('DEMO_INPUT=', DEMO_INPUT)


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


actual_sha = sha256_file(MODEL_PATH)
assert actual_sha == EXPECTED_SHA256
print('MODEL_SHA256=PASS')
print(actual_sha)


In [ ]:
import torch

checkpoint = torch.load(
    MODEL_PATH,
    map_location='cpu',
    weights_only=False,
)

assert checkpoint is not None
del checkpoint

print('TORCH_LOAD_MAP_LOCATION_CPU=PASS')
print('CUDA_VISIBLE_DEVICES=', os.environ['CUDA_VISIBLE_DEVICES'])


In [ ]:
# Segunda etapa (identificación) requiere cv2; se importa perezosamente
# dentro de src.recognition/src.predict -- este notebook sí lo necesita
# porque pasa --catalog.
from src.predict import ejecutar_inferencia

summary = ejecutar_inferencia(
    MODEL_PATH,
    DEMO_INPUT,
    OUTPUT_DIR,
    device='cpu',
    conf=0.25,
    limit=5,
    overwrite=True,
    catalog_path=CATALOG_PATH,
)

assert summary['device'] == 'cpu'
assert summary['model_sha256'] == EXPECTED_SHA256

print('CPU_INFERENCE_AND_IDENTIFICATION=PASS')
print('IMAGES=', summary['n_images'])
print(f"MEAN_INFERENCE_MS={summary['mean_inference_ms']:.3f}")


In [ ]:
from PIL import Image

annotated = sorted((OUTPUT_DIR / 'annotated').glob('*'))
assert len(annotated) == summary['n_images']

try:
    get_ipython
except NameError:
    running_in_notebook = False
else:
    running_in_notebook = True

if running_in_notebook:
    from IPython.display import display

    for image_path in annotated:
        with Image.open(image_path) as image:
            display(image.convert('RGB').copy())
else:
    for image_path in annotated:
        print('ANNOTATED_IMAGE=', image_path)

print('VISUALIZATION_READY=PASS')


## Tabla de resultados (detección + identidad)


In [ ]:
predictions_csv = OUTPUT_DIR / 'predictions.csv'
assert predictions_csv.is_file(), predictions_csv

with predictions_csv.open(newline='', encoding='utf-8') as handle:
    filas = list(csv.DictReader(handle))

header = ('imagen', 'bbox', 'yolo_conf', 'sku_id', 'product_name', 'identity_similarity')
print((' | '.join(header)))
for fila in filas:
    bbox = f"({fila['xmin']},{fila['ymin']},{fila['xmax']},{fila['ymax']})"
    print(' | '.join([
        fila['image'], bbox, f"{float(fila['yolo_confidence']):.3f}",
        fila['sku_id'], fila['product_name'][:40], fila['identity_similarity'],
    ]))

print()
print('N_PRODUCTOS_DETECTADOS=', len(filas))
print('N_UNKNOWN=', sum(1 for f in filas if f['sku_id'] == 'UNKNOWN'))


## Aciertos y errores de identidad (contra ground truth de composición, si está disponible)

Las escenas de demo son sintéticas y derivan del mismo catálogo de referencia usado para identificar: esta comparación es una evaluación **cerrada/optimista** (mide si el retrieval encuentra su propia fuente), no una prueba de generalización a fotografías nuevas de una caja real.


In [ ]:
import json

# El ground truth de composición ordena los objetos por z_order (tamaño de
# cutout descendente), NO por el orden en que YOLO reporta sus detecciones.
# Emparejar por índice de detección sería incorrecto: se empareja cada
# detección con la línea de ground truth de MAYOR IoU (>=0.5), igual que la
# evaluación end-to-end rigurosa (ver end_to_end_summary.json).
GROUND_TRUTH_PATH = Path(os.environ.get('AMARKET_GROUND_TRUTH', ''))
LABELS_DIR = DEMO_INPUT.parent / 'labels'


def iou_xyxy(a, b):
    ax0, ay0, ax1, ay1 = a
    bx0, by0, bx1, by1 = b
    ix0, iy0 = max(ax0, bx0), max(ay0, by0)
    ix1, iy1 = min(ax1, bx1), min(ay1, by1)
    iw, ih = max(0, ix1 - ix0), max(0, iy1 - iy0)
    inter = iw * ih
    area_a = max(0, ax1 - ax0) * max(0, ay1 - ay0)
    area_b = max(0, bx1 - bx0) * max(0, by1 - by0)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


if GROUND_TRUTH_PATH and GROUND_TRUTH_PATH.is_file() and LABELS_DIR.is_dir():
    from PIL import Image as _Image

    gt = json.loads(GROUND_TRUTH_PATH.read_text(encoding='utf-8'))
    for fila in filas:
        scene_id = Path(fila['image']).stem
        scene_gt = gt.get(scene_id, [])
        label_path = LABELS_DIR / f'{scene_id}.txt'
        det_box = (float(fila['xmin']), float(fila['ymin']), float(fila['xmax']), float(fila['ymax']))
        if not scene_gt or not label_path.is_file():
            print(f"SIN_GT  {fila['image']}  pred={fila['sku_id']}")
            continue
        with _Image.open(DEMO_INPUT / fila['image']) as im:
            W, H = im.size
        lines = [l for l in label_path.read_text(encoding='utf-8').splitlines() if l.strip()]
        best_iou, best_true_sku = 0.0, None
        for i, line in enumerate(lines):
            _, xc, yc, bw, bh = line.split()
            xc, yc, bw, bh = float(xc), float(yc), float(bw), float(bh)
            gt_box = ((xc - bw / 2) * W, (yc - bh / 2) * H, (xc + bw / 2) * W, (yc + bh / 2) * H)
            iou = iou_xyxy(det_box, gt_box)
            if iou > best_iou:
                best_iou, best_true_sku = iou, scene_gt[i]['sku_id']
        matched = best_iou >= 0.5
        acierto = matched and best_true_sku == fila['sku_id']
        marca = 'ACIERTO' if acierto else 'ERROR'
        print(f"{marca}  {fila['image']}  iou={best_iou:.2f}  pred={fila['sku_id']}  true={best_true_sku}")
else:
    print('GROUND_TRUTH/labels no disponibles en este entorno -- se omite la '
          'marca acierto/error; la tabla de arriba ya muestra la '
          'identificación real producida.')


## Resultado final — Experimento B (T1, ganador por mayor `val mAP50-95`)

### Detector (test AMARKET real, 98/98, una sola evaluación, sin tuning posterior)

| Métrica | Resultado |
|---|---:|
| Precision | 0.985658 |
| Recall | 1.000000 |
| F1 | 0.992777 |
| mAP@0.5 | 0.994091 |
| mAP@0.5:0.95 | 0.971511 |

### Identificación de producto (retrieval SIFT, evaluación end-to-end sobre 197 escenas sintéticas / 481 placements — ver `end_to_end_summary.json` para las cifras completas y su fecha de generación)

| Métrica | Resultado |
|---|---:|
| Detector recall (IoU>=0.5) | 1.000000 |
| Detector precision (IoU>=0.5) | 0.997925 |
| Identity top-1 accuracy (sobre detectados) | 0.846154 |
| Identity top-5 accuracy (sobre detectados) | 0.923077 |
| End-to-end accuracy (IoU correcto + SKU top-1 correcto) | 0.846154 |
| UNKNOWN rate | 0.000000 |
| Latencia media por escena | 45.24 s |

Base: 197 escenas / 481 placements reales (`end_to_end_summary.json`, `end_to_end_detail.csv`).

**Limitación explícita**: la identidad se mide en un dominio cerrado — las escenas sintéticas de evaluación derivan de los mismos cutouts que el catálogo de referencia. No es evidencia de generalización a fotografías nuevas de una caja de supermercado real.
